<a href="https://colab.research.google.com/github/buntyedutainer/student-performance-ai/blob/main/Notebook6_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Upload the file into Colab
from google.colab import files
files.upload()   # pick train.csv

Saving train.csv to train.csv


{'train.csv': b'school,sex,age,address,famsize,Pstatus,Medu,Fedu,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,Mjob_health,Mjob_other,Mjob_services,Mjob_teacher,Fjob_health,Fjob_other,Fjob_services,Fjob_teacher,reason_home,reason_other,reason_reputation,guardian_mother,guardian_other,G3\n0,1,19,0,1,1,3,3,1,2,1,0,0,0,1,1,1,0,1,4,5,3,1,2,5,0,15,12,0,1,0,0,0,0,1,0,0,0,1,0,0,12\n1,0,18,1,1,1,2,3,2,1,0,0,1,1,0,1,1,1,1,5,2,3,1,2,4,0,11,10,0,0,0,0,0,0,1,0,0,0,0,0,0,10\n0,0,19,0,1,1,2,3,1,3,1,0,0,0,0,1,1,1,1,4,1,2,1,1,3,20,13,11,0,1,0,0,0,1,0,0,0,0,1,0,1,11\n0,1,15,1,1,1,4,3,1,2,0,0,1,1,1,1,1,1,0,4,3,3,2,3,5,6,9,9,0,0,0,1,0,1,0,0,1,0,0,1,0,10\n0,1,17,1,0,0,4,1,2,1,0,0,0,1,1,1,1,1,1,4,5,4,2,4,5,20,8,8,0,0,1,0,0,1,0,0,1,0,0,1,0,8\n1,0,18,1,0,1,3,1,1,2,0,0,1,1,0,1,1,1,0,4,3,4,1,1,1,0,7,9,0,0,0,1,0,0,1,0,0,0,0,1,0,8\n0,0,16,1,1,1,4,4,1,3,0,0,1,1,1,1,1,1,0,3,2,3,1,2,2,6,13,14,0,0,1,0,0,0,1,0,0,0,

In [2]:
# Upload the file into Colab
from google.colab import files
files.upload()   # pick test.csv

Saving test.csv to test.csv


{'test.csv': b'school,sex,age,address,famsize,Pstatus,Medu,Fedu,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,Mjob_health,Mjob_other,Mjob_services,Mjob_teacher,Fjob_health,Fjob_other,Fjob_services,Fjob_teacher,reason_home,reason_other,reason_reputation,guardian_mother,guardian_other,G3\n0,1,16,1,1,1,4,4,1,1,0,0,0,0,1,1,1,1,0,5,3,2,1,2,5,0,13,12,0,0,1,0,0,0,1,0,0,0,0,1,0,12\n0,1,15,1,1,1,4,4,1,2,0,0,1,0,1,1,1,1,0,4,3,3,1,1,5,2,19,18,0,0,1,0,0,0,0,1,0,0,0,0,0,18\n0,0,18,1,1,1,4,4,1,2,0,0,1,1,0,1,1,1,0,4,4,4,3,3,5,2,11,11,0,0,0,1,0,1,0,0,0,0,0,1,0,11\n0,0,17,1,1,1,4,4,1,3,0,0,1,1,1,1,1,1,0,5,4,4,1,3,4,7,10,9,0,0,0,1,0,0,1,0,0,0,0,1,0,9\n0,0,15,1,1,0,4,3,1,2,0,0,1,1,1,1,1,1,0,4,3,2,1,1,1,0,14,15,0,0,1,0,0,0,1,0,0,0,1,1,0,15\n1,0,19,1,0,1,3,2,2,2,2,0,0,0,1,1,1,0,1,3,2,2,1,1,3,4,7,7,0,0,1,0,0,0,1,0,1,0,0,0,1,9\n0,0,16,1,1,1,4,4,1,1,0,0,1,0,0,1,1,1,0,4,4,4,1,2,2,4,14,14,1,0,0,0,0,1,0,0,1,0,

In [3]:
# Cell 1: setup
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
print("Train:", train.shape, "| Test:", test.shape)

Train: (285, 42) | Test: (72, 42)


In [4]:
%%writefile features.py
# Cell 2: one reusable function. Your Streamlit app will import this on Day 25.
import pandas as pd

def add_features(df):
    out = df.copy()
    out["avg_parent_edu"] = (out["Medu"] + out["Fedu"]) / 2
    out["support_score"] = out["famsup"] + out["schoolsup"] + out["paid"]
    out["alcohol_avg"] = (out["Dalc"] + out["Walc"]) / 2
    out["social_load"] = out["goout"] + out["freetime"]
    out["has_failures"] = (out["failures"] > 0).astype(int)
    out["no_absences"] = (out["absences"] == 0).astype(int)
    if {"G1", "G2"}.issubset(out.columns):
        out["grade_trend"] = out["G2"] - out["G1"]
    return out

Writing features.py


In [5]:
# Cell 3: apply the same function to train and test
from features import add_features

train_fe = add_features(train)
test_fe = add_features(test)

new_cols = [c for c in train_fe.columns if c not in train.columns]
print("New features:", new_cols)
print("Train shape now:", train_fe.shape, "| Test shape now:", test_fe.shape)
train_fe[new_cols].describe().round(2)

New features: ['avg_parent_edu', 'support_score', 'alcohol_avg', 'social_load', 'has_failures', 'no_absences', 'grade_trend']
Train shape now: (285, 49) | Test shape now: (72, 49)


,avg_parent_edu,support_score,alcohol_avg,social_load,has_failures,no_absences,grade_trend
count,285.00,285.00,285.00,285.00,285.00,285.00,285.00
mean,2.65,1.23,1.92,6.37,0.18,0.23,0.07
std,0.99,0.87,1.03,1.70,0.39,0.42,1.44
min,0.50,0.00,1.00,2.00,0.00,0.00,-4.00
25%,2.00,1.00,1.00,5.00,0.00,0.00,-1.00
50%,2.50,1.00,1.50,6.00,0.00,0.00,0.00
75%,3.50,2.00,2.50,7.00,0.00,0.00,1.00
max,4.00,3.00,5.00,10.00,1.00,1.00,4.00


In [6]:
# Cell 4: do the new features correlate with G3? (training data only)
corr_new = train_fe[new_cols + ["G3"]].corr()["G3"].drop("G3").sort_values()
print(corr_new.round(3))

print("\nFor comparison, the original columns they were built from:")
orig = ["Medu", "Fedu", "famsup", "schoolsup", "paid", "Dalc", "Walc",
        "goout", "freetime", "failures", "absences", "G1", "G2"]
print(train_fe[orig + ["G3"]].corr()["G3"].drop("G3").round(3).to_string())

has_failures     -0.275
alcohol_avg      -0.193
support_score    -0.135
social_load      -0.129
grade_trend       0.099
avg_parent_edu    0.174
no_absences       0.202
Name: G3, dtype: float64

For comparison, the original columns they were built from:
Medu         0.165
Fedu         0.151
famsup      -0.071
schoolsup   -0.223
paid        -0.010
Dalc        -0.149
Walc        -0.196
goout       -0.174
freetime    -0.033
failures    -0.292
absences    -0.229
G1           0.893
G2           0.966


In [7]:
# Cell 5: a helper that returns cross-validated RMSE on the training set
cv = KFold(n_splits=5, shuffle=True, random_state=42)

def cv_rmse(model_type, cols):
    X = train_fe[cols]
    y = train_fe["G3"]
    if model_type == "linear":
        scale_cols = [c for c in cols if X[c].nunique() > 2]
        model = Pipeline([
            ("prep", ColumnTransformer([("s", StandardScaler(), scale_cols)],
                                       remainder="passthrough")),
            ("model", LinearRegression())])
    else:
        model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    scores = -cross_val_score(model, X, y, cv=cv, scoring="neg_root_mean_squared_error")
    return scores.mean()

In [8]:
# Cell 6: define four feature sets
base_all = [c for c in train.columns if c != "G3"]
base_early = [c for c in base_all if c not in ("G1", "G2")]

engineered_early = [c for c in new_cols if c != "grade_trend"]
engineered_all = new_cols

sets = {
    "early, original": base_early,
    "early, + engineered": base_early + engineered_early,
    "all, original": base_all,
    "all, + engineered": base_all + engineered_all,
}
for name, cols in sets.items():
    print(f"{name:22s} {len(cols):3d} features")

early, original         39 features
early, + engineered     45 features
all, original           41 features
all, + engineered       48 features


In [9]:
# Cell 7: the comparison that matters, 5-fold CV RMSE (lower is better)
rows = []
for name, cols in sets.items():
    rows.append({
        "feature set": name,
        "features": len(cols),
        "Linear RMSE": round(cv_rmse("linear", cols), 3),
        "Random Forest RMSE": round(cv_rmse("forest", cols), 3),
    })
comparison = pd.DataFrame(rows)
comparison

,feature set,features,Linear RMSE,Random Forest RMSE
0,"early, original",39,3.052,3.024
1,"early, + engineered",45,3.079,3.045
2,"all, original",41,0.906,0.909
3,"all, + engineered",48,0.920,0.898


In [10]:
# Cell 8: why linear regression may not change, shown with a check
# avg_parent_edu is an exact combination of Medu and Fedu
check = train_fe["avg_parent_edu"] - (train_fe["Medu"] + train_fe["Fedu"]) / 2
print("Max difference between avg_parent_edu and (Medu+Fedu)/2:", check.abs().max())

# same for grade_trend
print("Max difference for grade_trend:",
      (train_fe["grade_trend"] - (train_fe["G2"] - train_fe["G1"])).abs().max())

# the linear model can therefore already express them
gains = comparison.set_index("feature set")
print("\nLinear change, early:", round(gains.loc["early, + engineered", "Linear RMSE"]
                                    - gains.loc["early, original", "Linear RMSE"], 3))
print("Linear change, all  :", round(gains.loc["all, + engineered", "Linear RMSE"]
                                    - gains.loc["all, original", "Linear RMSE"], 3))

Max difference between avg_parent_edu and (Medu+Fedu)/2: 0.0
Max difference for grade_trend: 0

Linear change, early: 0.027
Linear change, all  : 0.014


In [11]:
# Cell 9: an evidence-based decision rule
THRESHOLD = 0.05   # a feature set must improve CV RMSE by at least this much

decisions = []
for family, base, plus in [("early", "early, original", "early, + engineered"),
                           ("all", "all, original", "all, + engineered")]:
    for model in ["Linear RMSE", "Random Forest RMSE"]:
        gain = gains.loc[base, model] - gains.loc[plus, model]
        decisions.append({"set": family, "model": model.replace(" RMSE", ""),
                          "improvement": round(gain, 3),
                          "keep engineered?": "Yes" if gain >= THRESHOLD else "No"})
pd.DataFrame(decisions)

,set,model,improvement,keep engineered?
0,early,Linear,-0.027,No
1,early,Random Forest,-0.021,No
2,all,Linear,-0.014,No
3,all,Random Forest,0.011,No


In [12]:
# Cell 10: save the Phase 2 outputs (test set is only transformed, never used for tuning)
train_fe.to_csv("train_fe.csv", index=False)
test_fe.to_csv("test_fe.csv", index=False)
print("Saved train_fe.csv", train_fe.shape, "and test_fe.csv", test_fe.shape)

Saved train_fe.csv (285, 49) and test_fe.csv (72, 49)
